<a href="https://colab.research.google.com/github/Lornatt88/maai902-nlp-solution/blob/main/notebooks/05_rag_explanations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5: RAG explanations
Explains each flagged clause in plain English: the system first retrieves expert-written legal explanations, then an LLM writes the explanation using only those.

## 1. Building the knowledge base
Loads 79 expert explanations from Ruggeri et al. (2022) for five unfair types, each with a severity level. Adds three explanations for the remaining types (Jurisdiction, Choice of law, Contract by using), based on Lippi et al. (2019), Section 3.1.

In [3]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/federicoruggeri/Memnet_ToS/master/local_database/KB"
CODES = {"A": "Arbitration", "CH": "Unilateral change", "CR": "Content removal",
         "LTD": "Limitation of liability", "TER": "Unilateral termination"}

# Ruggeri et al. (2022) explanations
parts = []
for code, type_name in CODES.items():
    t = pd.read_csv(f"{BASE}/{code}_explanations.csv", index_col=0)
    t = t.rename(columns={"Id": "name", "Tag": "tag", "Explanation": "text"})
    t["type"] = type_name
    t["severity"] = t["tag"].str[-1].map({"2": "potentially unfair", "3": "clearly unfair"})
    t["source"] = "Ruggeri et al. (2022)"
    parts.append(t)

# Own explanations for the 3 missing types, based on Lippi et al. (2019), Section 3.1
own = pd.DataFrame([
    {"type": "Jurisdiction", "name": "court_away_from_residence", "severity": "clearly unfair",
     "text": "the clause requires any court case to take place away from the consumer's place of residence, in a different city or country, making it harder and more expensive to take legal action.",
     "source": "own, based on Lippi et al. (2019), Section 3.1 (ECJ case C-240/98)"},
    {"type": "Choice of law", "name": "law_not_of_residence", "severity": "potentially unfair",
     "text": "the clause makes a law other than the law of the consumer's country of residence govern the contract, which may reduce the legal protection the consumer would normally have.",
     "source": "own, based on Lippi et al. (2019), Section 3.1"},
    {"type": "Contract by using", "name": "use_is_consent", "severity": "potentially unfair",
     "text": "the clause binds the consumer to the terms simply by using the service, without requiring them to confirm that they have read and accepted the terms.",
     "source": "own, based on Lippi et al. (2019), Section 3.1"},
])

kb = pd.concat(parts + [own], ignore_index=True)
kb["text"] = (kb["text"].str.strip()
              .str.replace(r"^since\s+(the clause states that\s+)?", "", regex=True, case=False))
print("Explanations:", len(kb))
print(kb.groupby("type").size())

Explanations: 87
type
Arbitration                 8
Choice of law               1
Content removal            17
Contract by using           1
Jurisdiction                1
Limitation of liability    24
Unilateral change           8
Unilateral termination     27
dtype: int64


The knowledge base contains 87 explanations: 84 from the publicly released files of Ruggeri et al. (2022) for five unfair types (the paper reports 79, so the released files appear to be a slightly updated version), and 3 written for this project for Jurisdiction, Choice of law and Contract by using, based on the definitions and fairness rules in Lippi et al. (2019), Section 3.1.